# Silver Layer Transformation Notebook
## Purpose: Build final Silver tables with joins and DQ flags

**Note: Uses Spark tables (DBFS disabled in Free Edition)**

In [ ]:
# Silver Layer Transformation
from pyspark.sql.functions import col, when, coalesce, lit, hour, dayofweek, current_timestamp
from pyspark.sql.functions import count, sum as spark_sum, round as spark_round, first, max as spark_max
from pyspark.sql.window import Window

# spark session already exists in Databricks notebooks

print("Starting Silver layer transformation")

In [ ]:
# Build order_facts table
print("\n--- Building order_facts ---")

orders = spark.table("silver_orders_clean")
items_agg = spark.table("silver_order_items_agg")
refunds_agg = spark.table("silver_refunds_agg")

# Join all
order_facts = orders\
    .join(items_agg, on="order_id", how="left")\
    .join(refunds_agg, on="order_id", how="left")

# Fill nulls
order_facts = order_facts.fillna({
    "item_count": 0,
    "items_subtotal": 0.0,
    "has_refund": False,
    "refund_amount": 0.0
})

# Add derived columns
order_facts = order_facts\
    .withColumn("order_date", col("order_ts").cast("date"))\
    .withColumn("order_hour", hour(col("order_ts")))\
    .withColumn("time_slot",
        when(col("order_hour").between(8, 11), "morning")
        .when(col("order_hour").between(12, 15), "lunch")
        .when(col("order_hour").between(16, 19), "evening")
        .when(col("order_hour").between(20, 23), "dinner")
        .otherwise("late_night"))\
    .withColumn("day_of_week", dayofweek(col("order_ts")))\
    .withColumn("is_weekend", col("day_of_week").isin([1, 7]))

# DQ flags
order_facts = order_facts\
    .withColumn("_dq_has_null_customer", col("customer_id").isNull())\
    .withColumn("_dq_invalid_order_value", col("order_value").isNull() | (col("order_value") <= 0))

# Save
order_facts.write.mode("overwrite").partitionBy("order_date").saveAsTable("silver_order_facts")
print(f"✓ silver_order_facts: {order_facts.count()} rows")

In [ ]:
# Build delivery_ops table
print("\n--- Building delivery_ops ---")

events = spark.table("silver_delivery_events")
orders = spark.table("silver_orders_clean")

VALID_EVENT_TYPES = ["order_confirmed", "restaurant_accepted", "food_prep_started",
    "food_ready", "rider_assigned", "rider_picked_up", "out_for_delivery", 
    "delivered", "delivery_failed", "cancelled"]

# Check for orphan orders
order_ids = orders.select("order_id").distinct()
delivery_ops = events.join(order_ids, events["order_id"] == order_ids["order_id"], "left")\
    .withColumn("_dq_orphan_order", order_ids["order_id"].isNull())\
    .drop(order_ids["order_id"])

# DQ flags
delivery_ops = delivery_ops\
    .withColumn("_dq_invalid_event", ~col("event_type").isin(VALID_EVENT_TYPES))\
    .withColumn("_dq_missing_rider",
        col("event_type").isin(["rider_assigned", "rider_picked_up", "out_for_delivery", "delivered"])
        & col("rider_id").isNull())

delivery_ops.write.mode("overwrite").saveAsTable("silver_delivery_ops")
print(f"✓ silver_delivery_ops: {delivery_ops.count()} rows")

In [ ]:
# Build restaurant_support table
print("\n--- Building restaurant_support ---")

tickets = spark.table("silver_support_tickets")
restaurants = spark.table("silver_restaurants")

# Join tickets with restaurants
restaurant_support = tickets.join(
    restaurants.select("restaurant_id", "cuisine_type", "rating_band", "onboarding_date"),
    on="restaurant_id",
    how="left"
)

restaurant_support.write.mode("overwrite").saveAsTable("silver_restaurant_support")
print(f"✓ silver_restaurant_support: {restaurant_support.count()} rows")

In [ ]:
# Verify
print("\n" + "="*60)
print("SILVER LAYER COMPLETE")
print("="*60)

tables = ["silver_order_facts", "silver_delivery_ops", "silver_restaurant_support"]
for t in tables:
    count = spark.table(t).count()
    print(f"  {t}: {count:,} rows")

spark.stop()